# Hands-on Decision Tree: Beer Preference

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mojiahui777/ba_course/blob/main/decision_tree_beer_preference_demo.ipynb)

> **Goal:** Predict beer preference (`Regular` or `Light`) with a decision tree.
>
> Run the example first. Then change the data source, column lists, or row ranges and run all cells again.

## Learning objectives

- select historical rows and unlabeled future rows;
- train and evaluate a binary classification model;
- display probability and class predictions for test observations; and
- predict the class of new observations.

## 1. Import libraries

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    recall_score,
)
from sklearn.model_selection import train_test_split

## 2. Load the data

- Keep `UPLOAD_FILE = False` to load the Beer CSV from GitHub.
- Change it to `True` to upload a CSV in Colab.
- To use another online CSV, replace `DATA_URL`.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/mojiahui777/ba_course/main/WolfpackBeer_example.csv"
LOCAL_FILE = "WolfpackBeer_example.csv"
UPLOAD_FILE = False

if UPLOAD_FILE:
    from google.colab import files
    uploaded = files.upload()
    file_name = next(iter(uploaded))
    df = pd.read_csv(file_name)
else:
    try:
        df = pd.read_csv(DATA_URL)
    except Exception:
        df = pd.read_csv(LOCAL_FILE)

print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
display(df.head())

## 3. Specify columns, row ranges, and model settings

Row numbers below count data rows only; the CSV header is not row 1. Both range endpoints are included.

In [ ]:
TARGET_COLUMN = "Preference"
POSITIVE_CLASS = "Regular"

NUMERICAL_COLUMNS = ["Income", "Age"]
CATEGORICAL_COLUMNS = ["Gender", "Married"]

HISTORICAL_START_ROW = 1
HISTORICAL_END_ROW = 100
NEW_START_ROW = 101
NEW_END_ROW = 109

TEST_SIZE = 0.20
RANDOM_STATE = 42

## 4. Select historical and new observations

In [ ]:
historical = df.iloc[HISTORICAL_START_ROW - 1:HISTORICAL_END_ROW].copy()
new_observations = df.iloc[NEW_START_ROW - 1:NEW_END_ROW].copy()

if historical[TARGET_COLUMN].isna().any():
    raise ValueError("Every historical observation must have a known class.")
if not new_observations[TARGET_COLUMN].isna().all():
    raise ValueError("The new-observation target values must be blank.")

print(f"Historical observations: {len(historical)}")
print(f"New observations: {len(new_observations)}")
display(historical[TARGET_COLUMN].value_counts().rename("count").to_frame())

## 5. Prepare predictors and target

Only the listed numerical and categorical columns are used. Other CSV columns are ignored.

In [ ]:
PREDICTOR_COLUMNS = NUMERICAL_COLUMNS + CATEGORICAL_COLUMNS

X = historical[PREDICTOR_COLUMNS].copy()
X_new = new_observations[PREDICTOR_COLUMNS].copy()
y = historical[TARGET_COLUMN].copy()

X = pd.get_dummies(X, columns=CATEGORICAL_COLUMNS, drop_first=True, dtype=float)
X_new = pd.get_dummies(
    X_new, columns=CATEGORICAL_COLUMNS, drop_first=True, dtype=float
).reindex(columns=X.columns, fill_value=0)

classes = list(pd.unique(y))
if len(classes) != 2 or POSITIVE_CLASS not in classes:
    raise ValueError("The target must have two classes and include POSITIVE_CLASS.")
NEGATIVE_CLASS = next(value for value in classes if value != POSITIVE_CLASS)

display(X.head())

## 6. Create the training and testing datasets

In [ ]:
data_rows = pd.Series(historical.index + 1, index=historical.index, name="Data row")

X_train, X_test, y_train, y_test, row_train, row_test = train_test_split(
    X,
    y,
    data_rows,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Training observations: {len(X_train)}")
print(f"Testing observations: {len(X_test)}")

## 7. Train the decision tree

In [ ]:
MAX_DEPTH = 3

tree_model = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=MAX_DEPTH,
    random_state=RANDOM_STATE,
)
tree_model.fit(X_train, y_train)

positive_index = list(tree_model.classes_).index(POSITIVE_CLASS)
test_probability = np.clip(tree_model.predict_proba(X_test)[:, positive_index], 0, 1)
test_predicted_class = tree_model.predict(X_test)
new_probability = np.clip(tree_model.predict_proba(X_new)[:, positive_index], 0, 1)
new_predicted_class = tree_model.predict(X_new)

## 8. Evaluate the testing predictions

In [ ]:
metrics_table = pd.DataFrame(
    {
        "Metric": [
            "Accuracy",
            "Balanced accuracy",
            f"Sensitivity / recall ({POSITIVE_CLASS})",
            f"Specificity / recall ({NEGATIVE_CLASS})",
        ],
        "Value": [
            accuracy_score(y_test, test_predicted_class),
            balanced_accuracy_score(y_test, test_predicted_class),
            recall_score(y_test, test_predicted_class, pos_label=POSITIVE_CLASS),
            recall_score(y_test, test_predicted_class, pos_label=NEGATIVE_CLASS),
        ],
    }
)
display(metrics_table.round(3))

print("Classification report:")
print(classification_report(y_test, test_predicted_class, zero_division=0))

## 9. Display the confusion matrix

In [ ]:
class_order = [NEGATIVE_CLASS, POSITIVE_CLASS]
ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_predicted_class,
    labels=class_order,
    display_labels=class_order,
    cmap="Blues",
    colorbar=False,
)
plt.title("Testing Confusion Matrix")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.tight_layout()
plt.show()

## 10. Display probability and class predictions for the testing data

In [ ]:
probability_column = f"Predicted probability of {POSITIVE_CLASS}"
test_results = pd.DataFrame(
    {
        "Data row": row_test.to_numpy(),
        "True class": y_test.to_numpy(),
        probability_column: np.asarray(test_probability),
        "Predicted class": np.asarray(test_predicted_class),
    }
).sort_values("Data row").reset_index(drop=True)

display(test_results.round({probability_column: 3}))

## 11. Predict the new observations

In [ ]:
new_results = pd.DataFrame(
    {
        "Data row": new_observations.index + 1,
        probability_column: np.asarray(new_probability),
        "Predicted class": np.asarray(new_predicted_class),
    }
).sort_values("Data row").reset_index(drop=True)

display(new_results.round({probability_column: 3}))

## 12. Display the decision tree

In [ ]:
plt.figure(figsize=(18, 9))
plot_tree(
    tree_model,
    feature_names=list(X.columns),
    class_names=[str(value) for value in tree_model.classes_],
    filled=True,
    rounded=True,
    fontsize=9,
)
plt.title(f"Beer Preference Decision Tree (Maximum Depth {MAX_DEPTH})")
plt.tight_layout()
plt.show()

## 13. Display feature importance

In [ ]:
importance_table = pd.DataFrame(
    {
        "Variable": X.columns,
        "Importance": tree_model.feature_importances_,
    }
).sort_values("Importance", ascending=False)
display(importance_table.round(3))

importance_table.sort_values("Importance").plot(
    x="Variable", y="Importance", kind="barh", legend=False, color="#CC0000"
)
plt.title("Decision Tree Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 14. Validate the required outputs

In [ ]:
allowed_classes = {POSITIVE_CLASS, NEGATIVE_CLASS}

expected_historical_rows = HISTORICAL_END_ROW - HISTORICAL_START_ROW + 1
expected_new_rows = NEW_END_ROW - NEW_START_ROW + 1

assert len(historical) == expected_historical_rows
assert len(X_train) + len(X_test) == expected_historical_rows
assert len(new_observations) == expected_new_rows
assert len(test_results) == len(X_test)
assert len(new_results) == expected_new_rows
assert test_results[probability_column].between(0, 1).all()
assert new_results[probability_column].between(0, 1).all()
assert set(test_results["Predicted class"]).issubset(allowed_classes)
assert set(new_results["Predicted class"]).issubset(allowed_classes)

print("Validation passed.")